# RockID · Phase 2 · Training

Trains the rock classifier on the committed split manifest: MobileNetV2 (ImageNet), stage 1 with a frozen backbone, stage 2 fine-tuning the top ~30 %. Settings come from `ml/configs/train_v1.json`, so a re-run reproduces the result.

**Before you run**
1. **Runtime → Change runtime type → T4 GPU**.
2. Same Colab secrets as the audit (`GITHUB_TOKEN` if the repo is private; `KAGGLE_USERNAME` / `KAGGLE_KEY` only if `MyDrive/RockID/cache/dataset.zip` is gone).
3. The class list and split manifest are committed (`classes_v1.json`, `splits/split_manifest_classes_v1.csv`).

**Order of work.** Step 4 is the TF.js export smoke test: it must pass before training (Phase 2 acceptance). Set `SMOKE_TEST_ONLY = True` for a quick first run that stops right after it.

**Resume.** Everything lives in `MyDrive/RockID/runs/<run_name>/`. After a disconnect, **Runtime → Run all** continues the current stage from the last finished epoch; a finished stage is skipped.

This notebook only looks at **validation**. The test set, field set, calibration and the go/no-go gate are in `03_evaluate`.

## 1 · Settings

In [ ]:
REPO_URL = "https://github.com/GrowingRui/RockID.git"   # empty = use MyDrive/RockID/code/ml instead
REPO_BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/RockID"
CONFIG_FILE = "configs/train_v1.json"   # relative to ml/
RUN_SMOKE_TEST = True     # TF.js export check; must pass once per TF/converter combination
SMOKE_TEST_ONLY = False   # stop after the smoke test (first run)

## 2 · Setup: Drive, code, config

In [ ]:
import importlib, json, os, pathlib, shutil, subprocess, sys, time, zipfile
from datetime import datetime, timezone
from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = pathlib.Path(DRIVE_ROOT)
CACHE = DRIVE / "cache"
CACHE.mkdir(parents=True, exist_ok=True)

if REPO_URL:
    CODE = pathlib.Path("/content/rockid")
    url = REPO_URL
    try:
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        pass  # public repo, no token needed
    cmd = ["git", "-C", str(CODE), "pull", "--ff-only"] if CODE.exists() else \
          ["git", "clone", "--depth", "1", "-b", REPO_BRANCH, url, str(CODE)]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        raise RuntimeError("git clone/pull failed. Check REPO_URL, REPO_BRANCH and the GITHUB_TOKEN secret.") from None
    ML = CODE / "ml"
else:
    ML = DRIVE / "code" / "ml"

assert (ML / "rockid_ml").exists(), f"rockid_ml not found under {ML}"
sys.path.insert(0, str(ML))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagehash>=4.3"], check=True)  # not preinstalled on Colab

import numpy as np, pandas as pd
import tensorflow as tf
import rockid_ml
from rockid_ml import classes, images, metrics, prep, probes, report, train, tfjs_smoke
# Reload dependencies before the modules that use them, so a re-run after `git pull`
# never mixes old and new code in one session.
for m in (images, report, probes, classes, metrics, prep, train, tfjs_smoke):
    importlib.reload(m)

cfg = train.load_config(ML / CONFIG_FILE)
RUN = DRIVE / "runs" / cfg.run_name
train.prepare_run_dir(RUN, cfg)
tf.keras.utils.set_random_seed(cfg.seed)
if cfg.deterministic_ops:
    tf.config.experimental.enable_op_determinism()

print("rockid_ml", rockid_ml.__version__, "from", ML, "· commit",
      subprocess.run(["git", "-C", str(ML), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip() or "n/a")
print("TF", tf.__version__, "· GPU:", tf.config.list_physical_devices("GPU") or "NONE, switch to a T4 runtime")
print("run folder:", RUN)

## 3 · Images: manifest, dataset, prepared copies (cached on Drive)

Each image is decoded once with the audit's loader and checked against the manifest's pixel SHA-256, so training provably uses the exact audited images. Small JPEG copies (short side 288 px) are zipped to Drive; later runs only unzip them.

In [ ]:
manifest = pd.read_csv(ML / cfg.split_manifest)
cc = classes.load_class_config(ML / cfg.class_file)
stale = sorted(set(manifest["label"]) & (set(cc["merges"]) | set(cc["drop"])))
assert not stale, f"manifest has labels the class file merges or drops: {stale}. Re-run the audit split."
LABELS = sorted(manifest["label"].unique())
LABEL_ID = {l: i for i, l in enumerate(LABELS)}
print(f"{len(manifest):,} images · {len(LABELS)} classes · class list {cc.get('version')} ({cc.get('status')})")

WORK = pathlib.Path("/content/data"); WORK.mkdir(parents=True, exist_ok=True)
PREP = WORK / f"prep_{cfg.prep_short_side}"
PREP_ZIP = CACHE / f"prep_{cfg.prep_short_side}.zip"

def have_all(root):
    return all((root / prep.prep_name(p)).exists() for p in manifest["rel_path"])

if not have_all(PREP) and PREP_ZIP.exists():
    with zipfile.ZipFile(PREP_ZIP) as z:
        z.extractall(PREP)

if not have_all(PREP):
    DATA = WORK / "raw"
    ZIP_CACHE = CACHE / "dataset.zip"
    if not ZIP_CACHE.exists():
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME").strip()
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY").strip()
        subprocess.run(["kaggle", "datasets", "download", "-d", "kutlayusta/type-of-rocks-and-minerals-dataset",
                        "-p", str(WORK), "--force"], check=True)
        shutil.copy(next(WORK.glob("*.zip")), ZIP_CACHE)
    if not DATA.exists():
        with zipfile.ZipFile(ZIP_CACHE) as z:
            z.extractall(DATA)
    missing = [p for p in manifest["rel_path"] if not (DATA / p).exists()]
    assert not missing, f"{len(missing)} manifest files missing from the dataset, e.g. {missing[:3]}"

    checked = prep.prepare_images(manifest, DATA, PREP, short_side=cfg.prep_short_side)
    bad = checked[checked["sha_ok"] == False]  # noqa: E712 (nullable boolean)
    if len(bad):
        shutil.rmtree(PREP)
        raise RuntimeError(f"{len(bad)} images differ from the audited pixels, e.g. {bad['rel_path'].head(3).tolist()}. "
                           "The dataset changed since the audit: re-run 01_audit.")
    print(f"pixel check passed on {int(checked['sha_ok'].notna().sum()):,} images")
    tmp = WORK / PREP_ZIP.name
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED) as z:  # JPEGs don't compress further
        for p in manifest["rel_path"]:
            z.write(PREP / prep.prep_name(p), prep.prep_name(p))
    shutil.copy(tmp, PREP_ZIP)

manifest["prep_path"] = [str(PREP / prep.prep_name(p)) for p in manifest["rel_path"]]
manifest["y"] = manifest["label"].map(LABEL_ID)
splits = {s: g.reset_index(drop=True) for s, g in manifest.groupby("split")}
pd.crosstab(manifest["label"], manifest["split"])[["train", "val", "test"]].T

## 4 · TF.js export smoke test (before training)

Exports an **untrained** model (same architecture) to a SavedModel, converts it to an fp16 TF.js graph model (converter installed with `--no-deps`, so it runs on this TensorFlow and never replaces it), runs it in Node with `@tensorflow/tfjs` and compares the output with Keras. The versions that pass are written to `versions.json`: pin them in `ml/pyproject.toml` and keep them for Phase 3.

If it fails, the error shows which step broke. Usual fix: pick a `tfjs_converter` version that supports this TensorFlow, or pin TensorFlow to the last version that passed (`versions.json`).

In [ ]:
SMOKE = RUN / "smoke_test.json"
if RUN_SMOKE_TEST:
    prev = json.loads(SMOKE.read_text()) if SMOKE.exists() else None
    if prev and prev["ok"] and prev["versions"]["tensorflow"] == tf.__version__:
        smoke = prev
        print("smoke test already passed with TF", tf.__version__)
    else:
        m0, _ = train.build_model(cfg, len(LABELS))
        smoke = tfjs_smoke.run_smoke_test(m0, WORK / "smoke", cfg, ML / "scripts")
        SMOKE.write_text(json.dumps(smoke, indent=2))
        (RUN / "versions.json").write_text(json.dumps(smoke["versions"], indent=2))
        del m0
    print(json.dumps(smoke, indent=2))
    if not smoke["ok"]:
        raise RuntimeError("TF.js smoke test failed (see above). Fix the versions before training.")
if SMOKE_TEST_ONLY:
    raise SystemExit("SMOKE_TEST_ONLY: stopping here. Set it to False to train.")

## 5 · Datasets, class weights, model

In [ ]:
tf.keras.utils.set_random_seed(cfg.seed)  # same initial weights whether or not the smoke test ran
tr, va = splits["train"], splits["val"]
cw = metrics.class_weights(tr["y"].to_numpy(), len(LABELS), cfg.class_weight_power)
train_ds = train.make_dataset(tr["prep_path"], tr["y"], len(LABELS), cfg, training=True, sample_weight=cw[tr["y"]])
val_ds = train.make_dataset(va["prep_path"], va["y"], len(LABELS), cfg, training=False)
STEPS = train.steps_per_epoch(len(tr), cfg)

model, base = train.build_model(cfg, len(LABELS))
print(f"train {len(tr):,} · val {len(va):,} · class weights {cw.min():.2f}–{cw.max():.2f}")
print("params:", train.count_params(model))

# Look at a few augmented training images (shown here only, not saved)
import matplotlib.pyplot as plt
xb, yb, _ = next(iter(train_ds))
fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for ax, x, y in zip(axes.flat, xb.numpy(), yb.numpy()):
    ax.imshow(x.astype("uint8")); ax.set_title(LABELS[y.argmax()], fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 6 · Stage 1: frozen backbone

In [ ]:
t0 = time.time()
train.compile_model(model, cfg.stage1_lr, cfg)
h1 = train.fit_stage(model, train_ds, val_ds, cfg.stage1_epochs, STEPS, RUN / "stage1", cfg)
print(f"stage 1 best {cfg.monitor}: {h1[cfg.monitor].max():.3f} after {len(h1)} epochs")

## 7 · Stage 2: fine-tune the top of the backbone

In [ ]:
cut = train.unfreeze_top(base, cfg.unfreeze_frac)
train.compile_model(model, cfg.stage2_lr, cfg)
print(f"unfrozen backbone layers {cut}–{len(base.layers) - 1} (BatchNorm frozen) · params:", train.count_params(model))
h2 = train.fit_stage(model, train_ds, val_ds, cfg.stage2_epochs, STEPS, RUN / "stage2", cfg)
train_minutes = (time.time() - t0) / 60
print(f"stage 2 best {cfg.monitor}: {h2[cfg.monitor].max():.3f} after {len(h2)} epochs")

## 8 · Validation results and saved model

Saves `model.keras` (logits output), `labels.json` and the validation logits that `03_evaluate` uses to fit the temperature and the "uncertain" thresholds.

In [ ]:
val_logits = train.predict_logits(model, val_ds)
val = metrics.summarize(va["y"].to_numpy(), val_logits, LABELS)
print(f"val top-1 {val['top1']:.1%} · top-3 {val['top3']:.1%} · lowest recall {val['min_recall']:.0%} ({val['min_recall_class']})")

table = pd.DataFrame({"recall": val["per_class_recall"],
                      "val_images": va["label"].value_counts().reindex(LABELS),
                      "train_images": tr["label"].value_counts().reindex(LABELS),
                      "class_weight": cw}, index=LABELS).sort_values("recall")
confused = probes.confused_pairs(va["label"].to_numpy(), np.asarray(LABELS)[val["pred"]])

model.save(RUN / "model.keras")
(RUN / "labels.json").write_text(json.dumps(LABELS, indent=2))
np.savez_compressed(RUN / "val_logits.npz", logits=val_logits, y=va["y"].to_numpy(), rel_path=va["rel_path"].to_numpy())
table

## 9 · Report and download

In [ ]:
FIG = RUN / "figures"; FIG.mkdir(exist_ok=True)
figures = {"training curves": FIG / "history.png", "val recall per class": FIG / "val_recall.png"}
train.plot_history({"stage1": h1, "stage2": h2}, figures["training curves"])
report.plot_share(table, "recall", figures["val recall per class"], "validation recall", metrics.GATE["min_recall"])

done1, done2 = (json.loads((RUN / s / "done.json").read_text()) for s in ("stage1", "stage2"))
info = {
    "Data": f"{len(manifest):,} images, {len(LABELS)} classes ({cfg.split_manifest}, class list {cc.get('version')})",
    "Splits": f"train {len(tr):,} · val {len(va):,} · test {len(splits['test']):,} (test not used here)",
    "Model": f"{cfg.arch} α {cfg.alpha}, {cfg.image_size} px, logits output, in-graph normalization",
    "Stage 1": f"{done1['epochs_run']} epochs, best epoch {done1['best_epoch']} ({cfg.monitor} {done1[cfg.monitor]:.3f})",
    "Stage 2": f"{done2['epochs_run']} epochs, best epoch {done2['best_epoch']} ({cfg.monitor} {done2[cfg.monitor]:.3f})",
    "Training time (this session)": f"{train_minutes:.0f} min",
    "TF.js smoke test": "not run" if not RUN_SMOKE_TEST else
        f"{'passed' if smoke['ok'] else 'FAILED'} · {smoke['quantization']} · {smoke['size_mb']} MB · max |Δp| {smoke.get('max_abs_diff', float('nan')):.4f}",
    "Versions": json.dumps(smoke["versions"]) if RUN_SMOKE_TEST else f"TF {tf.__version__}",
}
path = train.write_report(RUN, cfg, info, val, table, confused, figures)
table.to_csv(RUN / "val_per_class.csv")
print("Report:", path)

from google.colab import files
bundle = WORK / f"rockid_train_{cfg.run_name}.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    base_dir = f"reports/train-{cfg.run_name}"
    z.write(path, f"{base_dir}/train.md")
    z.write(RUN / "val_per_class.csv", f"{base_dir}/val_per_class.csv")
    for f in FIG.glob("*.png"):
        z.write(f, f"{base_dir}/figures/{f.name}")
    for f in ("versions.json", "smoke_test.json"):
        if (RUN / f).exists():
            z.write(RUN / f, f"{base_dir}/{f}")
files.download(str(bundle))
print("Unzip into the repo's ml/ folder. The model itself stays on Drive until Phase 3 releases it.")

## Next
1. Commit `reports/train-<run_name>/` and pin the versions from `versions.json` in `ml/pyproject.toml`.
2. Run `03_evaluate` (test + field set, temperature scaling, thresholds, gate).
3. Reproducibility check: re-run this notebook with a new `run_name` (nothing else changed); top-1 should land within ±1.5 points.